# Phase 7 (Prior Case Retrieval) - data prep
Pull gated HF data, clean, audit, analyse, run lexical baselines, tokenise with the Phase-1 tokenizer, mine hard negatives, build folds.

**Before running:** Internet ON, paste your HF token into `HF_TOKEN` in section 1, attach the Phase-1 tokenizer dataset.

**Output:** `/kaggle/working/pcr_data_clean/`

## 1. Configuration

In [ ]:
import os
import re
import sys
import json
import glob
import time
import pickle
import shutil
import hashlib
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SPLITS = ["train", "dev", "test"]

# Paste your Hugging Face token here (hardcoded, no Kaggle secrets).
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

# Env vars are only for local testing. On Kaggle leave them alone.
LOCAL_DATA_DIR = os.environ.get("PCR_LOCAL_DATA_DIR", None)
TOKENIZER_DIR = os.environ.get("PCR_TOKENIZER_DIR", None)
OUT_DIR = os.environ.get("PCR_OUT_DIR", "/kaggle/working/pcr_data_clean")

SEED = 42
N_FOLDS = 5
TOP_K_EVAL = 7              # the competition metric is Micro-F1@7

# Cleaning
MIN_QUERY_WORDS = 20        # train/dev queries shorter than this are dropped
EMPTY_PLACEHOLDER = "."     # used when a test query / any candidate is empty (never dropped)

# Leakage handling (same idea as Phase 4/5/6)
EXCLUDE_TEST_OVERLAP_FROM_TRAIN = True   # drop train/dev queries whose id or text also appears as a test query

# Lexical baseline / hard negatives
BM25_K1 = 1.5
BM25_B = 0.75
BM25_TOP_N = 100            # ranked candidates saved per query (hard negatives + rerank pool)
BM25_QUERY_BATCH = 64

# Long-document views
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128
MAX_CHUNKS = 16
VIEW_WORDS = 1024           # word budget used in the head / tail / head+tail view analysis
ANALYSIS_QUERIES = 300      # dev queries used for the view analysis

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
print("OUT_DIR:", OUT_DIR)

## 2. Hugging Face login

In [ ]:
if LOCAL_DATA_DIR is None:
    if HF_TOKEN is None or HF_TOKEN.startswith("PASTE_"):
        raise RuntimeError("Set HF_TOKEN in the configuration cell (section 1).")

    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("Local data mode, skipping Hugging Face login.")

## 3. Find and download the PCR files (queries + candidate pool for every split)

In [ ]:
RAW_DIR = os.path.join(OUT_DIR, "raw_snapshot")
os.makedirs(RAW_DIR, exist_ok=True)

STEM_TO_SPLIT = {
    "train": "train",
    "dev": "dev",
    "val": "dev",
    "valid": "dev",
    "validation": "dev",
    "test": "test",
}


def simple_name(text):
    name = text.lower()
    name = name.replace("_", "")
    name = name.replace("-", "")
    name = name.replace(" ", "")
    return name


def is_pcr_path(path):
    key = simple_name(path)
    if "pcr" in key:
        return True
    if "priorcase" in key:
        return True
    if "phase7" in key:
        return True
    return False


def classify_file(path):
    # returns ("q" or "c", split) or None
    base = os.path.basename(path)
    stem, ext = os.path.splitext(base.lower())
    if ext not in [".json", ".jsonl"]:
        return None
    s = simple_name(stem)
    kind = "q"
    for word in ["candidates", "candidate", "cands", "pool"]:
        if word in s:
            kind = "c"
            s = s.replace(word, "")
            break
    split = None
    if s in STEM_TO_SPLIT:
        split = STEM_TO_SPLIT[s]
    else:
        parent = simple_name(os.path.basename(os.path.dirname(path)))
        if parent in STEM_TO_SPLIT and s in ["", "queries", "query"]:
            split = STEM_TO_SPLIT[parent]
    if split is None:
        return None
    return (kind, split)


found = {}
all_files = []

if LOCAL_DATA_DIR is not None:
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            all_files.append(full_path)
else:
    from huggingface_hub import list_repo_files
    all_files = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    print("Files in repo:", len(all_files))

for path in sorted(all_files, key=len):
    if not is_pcr_path(path):
        continue
    info = classify_file(path)
    if info is None:
        continue
    if info in found:
        print("  (ignoring extra match for %s: %s)" % (str(info), path))
        continue
    found[info] = path

for split in SPLITS:
    print(split, "queries   ->", found.get(("q", split)))
    print(split, "candidates->", found.get(("c", split)))

if ("q", "train") not in found or ("c", "train") not in found:
    print("\nAll repo files, to help you spot the right ones:")
    for p in all_files:
        print("  ", p)
    raise FileNotFoundError("No train query / candidate file found for the PCR phase.")

LOCAL_PATHS = {}
for info in found:
    path = found[info]
    if LOCAL_DATA_DIR is not None:
        LOCAL_PATHS[info] = path
    else:
        from huggingface_hub import hf_hub_download
        LOCAL_PATHS[info] = hf_hub_download(HF_DATASET_REPO, filename=path, repo_type="dataset",
                                            token=HF_TOKEN, local_dir=RAW_DIR)
        print("downloaded", path, "->", LOCAL_PATHS[info], "| %.1f MB" % (os.path.getsize(LOCAL_PATHS[info]) / 1e6))

PRESENT_SPLITS = []
for split in SPLITS:
    if ("q", split) in LOCAL_PATHS and ("c", split) in LOCAL_PATHS:
        PRESENT_SPLITS.append(split)
    else:
        print("NOTE: split '%s' is incomplete (needs both query and candidate file). Re-run when it is released." % split)
print("Splits in use:", PRESENT_SPLITS)

## 4. Load raw files
Handles dict, list-of-records and jsonl layouts. Ids are always kept as strings (leading zeros matter).

In [ ]:
ID_KEYS = ["id", "case_id", "query_id", "doc_id", "cand_id"]
TEXT_KEYS = ["text", "sentences", "document", "content"]
REL_KEYS = ["relevant_candidates", "relevant", "relevant_cases", "labels"]


def load_json_any(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    try:
        obj = json.loads(raw)
    except Exception:
        obj = []
        for line in raw.splitlines():
            line = line.strip()
            if len(line) == 0:
                continue
            obj.append(json.loads(line))
    records = []
    if isinstance(obj, dict):
        for key in obj:
            records.append((str(key), obj[key]))
    else:
        for item in obj:
            rec_id = None
            for k in ID_KEYS:
                if k in item:
                    rec_id = str(item[k])
                    break
            if rec_id is None:
                raise ValueError("Record without an id key in " + path)
            records.append((rec_id, item))
    return records


def get_sentences(rec):
    value = None
    if isinstance(rec, dict):
        for k in TEXT_KEYS:
            if k in rec:
                value = rec[k]
                break
    else:
        value = rec
    if value is None:
        return []
    if isinstance(value, str):
        return value.split("\n")
    sentences = []
    for s in value:
        if isinstance(s, str):
            sentences.append(s)
        elif s is None:
            sentences.append("")
        else:
            sentences.append(str(s))
    return sentences


def get_relevant(rec):
    if not isinstance(rec, dict):
        return None
    for k in REL_KEYS:
        if k in rec:
            value = rec[k]
            if value is None:
                return []
            if isinstance(value, (str, int)):
                return [str(value)]
            out = []
            for v in value:
                out.append(str(v))
            return out
    return None


RAW = {}
DUP_ID_COUNTS = {}
for split in PRESENT_SPLITS:
    RAW[split] = {"queries": {}, "cands": {}}
    for kind, name in [("q", "queries"), ("c", "cands")]:
        records = load_json_any(LOCAL_PATHS[(kind, split)])
        dup = 0
        for rec_id, rec in records:
            if rec_id in RAW[split][name]:
                dup += 1
                continue
            entry = {}
            entry["sentences"] = get_sentences(rec)
            if kind == "q":
                entry["relevant"] = get_relevant(rec)
            RAW[split][name][rec_id] = entry
        DUP_ID_COUNTS[(split, name)] = dup
    print("%-5s queries: %6d | candidates: %6d" % (split, len(RAW[split]["queries"]), len(RAW[split]["cands"])))

## 5. Raw audit
Looks for every anomaly that could hurt training or silently break the submission.

In [ ]:
PLACEHOLDER_RE = re.compile(r"<[A-Za-z_ ]{2,20}>")


def text_hash(sentences):
    joined = " ".join(sentences)
    joined = re.sub(r"\s+", " ", joined).strip().lower()
    return hashlib.md5(joined.encode("utf-8")).hexdigest()


AUDIT = {}
for split in PRESENT_SPLITS:
    queries = RAW[split]["queries"]
    cands = RAW[split]["cands"]
    a = {}

    a["dup_query_ids"] = DUP_ID_COUNTS[(split, "queries")]
    a["dup_cand_ids"] = DUP_ID_COUNTS[(split, "cands")]

    empty_q = 0
    empty_c = 0
    blank_sent = 0
    total_sent = 0
    placeholder_hits = Counter()
    for qid in queries:
        sents = queries[qid]["sentences"]
        if len(" ".join(sents).strip()) == 0:
            empty_q += 1
        for s in sents:
            total_sent += 1
            if len(s.strip()) == 0:
                blank_sent += 1
            for m in PLACEHOLDER_RE.findall(s):
                placeholder_hits[m] += 1
    for cid in cands:
        sents = cands[cid]["sentences"]
        if len(" ".join(sents).strip()) == 0:
            empty_c += 1
        for s in sents:
            total_sent += 1
            if len(s.strip()) == 0:
                blank_sent += 1
            for m in PLACEHOLDER_RE.findall(s):
                placeholder_hits[m] += 1
    a["empty_queries"] = empty_q
    a["empty_cands"] = empty_c
    a["blank_sentence_frac"] = blank_sent / max(total_sent, 1)
    a["placeholders"] = placeholder_hits.most_common(8)

    # label checks
    no_rel_key = 0
    zero_rel = 0
    rel_dups = 0
    rel_missing = 0
    rel_total = 0
    for qid in queries:
        rel = queries[qid]["relevant"]
        if rel is None:
            no_rel_key += 1
            continue
        if len(rel) == 0:
            zero_rel += 1
        if len(set(rel)) != len(rel):
            rel_dups += 1
        for r in rel:
            rel_total += 1
            if r not in cands:
                rel_missing += 1
    a["queries_without_label_field"] = no_rel_key
    a["queries_with_zero_relevant"] = zero_rel
    a["queries_with_duplicate_relevant"] = rel_dups
    a["relevant_ids_not_in_pool"] = rel_missing
    a["relevant_ids_total"] = rel_total

    # query inside its own pool
    in_pool = 0
    for qid in queries:
        if qid in cands:
            in_pool += 1
    a["query_id_in_own_pool"] = in_pool

    # identical text between a query and a candidate of its pool (different id)
    cand_hash_to_ids = defaultdict(list)
    for cid in cands:
        cand_hash_to_ids[text_hash(cands[cid]["sentences"])].append(cid)
    same_text_cand = 0
    for qid in queries:
        h = text_hash(queries[qid]["sentences"])
        for cid in cand_hash_to_ids.get(h, []):
            if cid != qid:
                same_text_cand += 1
                break
    a["queries_with_identical_text_candidate"] = same_text_cand

    dup_groups = 0
    dup_docs = 0
    for h in cand_hash_to_ids:
        if len(cand_hash_to_ids[h]) > 1:
            dup_groups += 1
            dup_docs += len(cand_hash_to_ids[h])
    a["dup_text_groups_in_pool"] = dup_groups
    a["dup_text_docs_in_pool"] = dup_docs

    id_lengths = Counter()
    for cid in cands:
        id_lengths[len(cid)] += 1
    a["cand_id_lengths"] = dict(id_lengths)

    AUDIT[split] = a

for split in PRESENT_SPLITS:
    print("=" * 70)
    print(split.upper())
    for key in AUDIT[split]:
        print("  %-38s %s" % (key, AUDIT[split][key]))

## 6. Cleaning
Length-safe text cleaning, junk-sentence removal, label repair. Candidates are **never** dropped (the pool must stay intact). Test queries are never dropped.

In [ ]:
CHAR_MAP = {}
for cp in list(range(0, 9)) + [11, 12] + list(range(14, 32)) + [127]:
    CHAR_MAP[cp] = " "
for cp in [0x200B, 0x200C, 0x200D, 0xFEFF, 0x00AD]:
    CHAR_MAP[cp] = None
for cp in [0x00A0, 0x2028, 0x2029, 0x0085, 0x2002, 0x2003, 0x2009]:
    CHAR_MAP[cp] = " "


def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(CHAR_MAP)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def has_alnum(text):
    for ch in text:
        if ch.isalnum():
            return True
    return False


CLEAN = {}
CLEAN_LOG = {}
for split in PRESENT_SPLITS:
    CLEAN[split] = {"queries": {}, "cands": {}}
    log = Counter()

    # candidates: keep all
    for cid in RAW[split]["cands"]:
        sents = []
        for s in RAW[split]["cands"][cid]["sentences"]:
            s2 = clean_text(s)
            if len(s2) == 0 or not has_alnum(s2):
                log["junk_sentences_removed"] += 1
                continue
            sents.append(s2)
        if len(sents) == 0:
            sents = [EMPTY_PLACEHOLDER]
            log["empty_candidates_kept_with_placeholder"] += 1
        CLEAN[split]["cands"][cid] = {"sentences": sents}

    pool = CLEAN[split]["cands"]
    for qid in RAW[split]["queries"]:
        raw_q = RAW[split]["queries"][qid]
        sents = []
        for s in raw_q["sentences"]:
            s2 = clean_text(s)
            if len(s2) == 0 or not has_alnum(s2):
                log["junk_sentences_removed"] += 1
                continue
            sents.append(s2)
        n_words = 0
        for s in sents:
            n_words += len(s.split())

        relevant = raw_q["relevant"]
        if split != "test":
            if relevant is None:
                log["dropped_query_no_label_field"] += 1
                continue
            if n_words < MIN_QUERY_WORDS:
                log["dropped_query_too_short"] += 1
                continue
            seen = set()
            kept = []
            for r in relevant:
                if r in seen:
                    log["duplicate_relevant_removed"] += 1
                    continue
                seen.add(r)
                if r not in pool:
                    log["relevant_not_in_pool_removed"] += 1
                    continue
                kept.append(r)
            if len(kept) == 0:
                log["dropped_query_no_relevant_left"] += 1
                continue
            CLEAN[split]["queries"][qid] = {"sentences": sents, "relevant": kept}
        else:
            if len(sents) == 0:
                sents = [EMPTY_PLACEHOLDER]
                log["empty_test_queries_kept_with_placeholder"] += 1
            CLEAN[split]["queries"][qid] = {"sentences": sents, "relevant": None}
    CLEAN_LOG[split] = log
    print(split, "-> queries kept %d / %d | candidates %d" % (
        len(CLEAN[split]["queries"]), len(RAW[split]["queries"]), len(CLEAN[split]["cands"])))
    for key in log:
        print("    %-45s %d" % (key, log[key]))

## 7. Cross-split leakage
Checks id / text overlap between splits and handles overlap with the **test queries** (same decision as Phase 4/5/6).

In [ ]:
def q_hash_map(split):
    out = {}
    for qid in CLEAN[split]["queries"]:
        out[qid] = text_hash(CLEAN[split]["queries"][qid]["sentences"])
    return out


LEAK = {}
if "test" in CLEAN:
    test_ids = set(CLEAN["test"]["queries"].keys())
    test_hashes = set(q_hash_map("test").values())
    for split in ["train", "dev"]:
        if split not in CLEAN:
            continue
        hashes = q_hash_map(split)
        id_overlap = []
        text_overlap = []
        for qid in CLEAN[split]["queries"]:
            if qid in test_ids:
                id_overlap.append(qid)
            if hashes[qid] in test_hashes:
                text_overlap.append(qid)
        LEAK[split] = {"id": id_overlap, "text": text_overlap}
        print("%s queries sharing id with a test query: %d | sharing text: %d" % (split, len(id_overlap), len(text_overlap)))

    if EXCLUDE_TEST_OVERLAP_FROM_TRAIN:
        for split in LEAK:
            drop = set(LEAK[split]["id"]) | set(LEAK[split]["text"])
            for qid in drop:
                del CLEAN[split]["queries"][qid]
            print("  removed %d %s queries (overlap with test)" % (len(drop), split))
else:
    print("No test split yet, skipping the test-overlap check.")

# overlap between splits at the pool level (informational)
print()
for a_name in PRESENT_SPLITS:
    for b_name in PRESENT_SPLITS:
        if a_name >= b_name:
            continue
        pool_a = set(CLEAN[a_name]["cands"].keys())
        pool_b = set(CLEAN[b_name]["cands"].keys())
        qa = set(CLEAN[a_name]["queries"].keys())
        qb = set(CLEAN[b_name]["queries"].keys())
        print("pool(%s) & pool(%s): %d | query(%s) in pool(%s): %d | query(%s) in pool(%s): %d" % (
            a_name, b_name, len(pool_a & pool_b),
            a_name, b_name, len(qa & pool_b),
            b_name, a_name, len(qb & pool_a)))

# train-gold candidates that also sit in the test pool (informational only, labels are NOT copied)
if "test" in CLEAN and "train" in CLEAN:
    train_gold = set()
    for qid in CLEAN["train"]["queries"]:
        for r in CLEAN["train"]["queries"][qid]["relevant"]:
            train_gold.add(r)
    print("train-gold candidates present in the test pool:", len(train_gold & set(CLEAN["test"]["cands"].keys())))

## 8. Basic analysis: sizes, label counts, lengths, candidate popularity

In [ ]:
import matplotlib
import matplotlib.pyplot as plt


def words_of_doc(sentences):
    n = 0
    for s in sentences:
        n += len(s.split())
    return n


def describe(values, name):
    arr = np.array(values, dtype=np.float64)
    if len(arr) == 0:
        print("  %-28s (empty)" % name)
        return
    print("  %-28s n=%d mean=%.0f median=%.0f p90=%.0f p99=%.0f max=%.0f" % (
        name, len(arr), arr.mean(), np.median(arr), np.percentile(arr, 90), np.percentile(arr, 99), arr.max()))


STATS = {}
for split in PRESENT_SPLITS:
    q_words = []
    for qid in CLEAN[split]["queries"]:
        q_words.append(words_of_doc(CLEAN[split]["queries"][qid]["sentences"]))
    c_words = []
    for cid in CLEAN[split]["cands"]:
        c_words.append(words_of_doc(CLEAN[split]["cands"][cid]["sentences"]))
    STATS[split] = {"q_words": q_words, "c_words": c_words}
    print(split.upper(), "| queries", len(q_words), "| pool size", len(c_words))
    describe(q_words, "query words")
    describe(c_words, "candidate words")

# relevant-count distribution (this decides how good F1@7 can get)
print("\nrelevant candidates per query")
for split in ["train", "dev"]:
    if split not in CLEAN:
        continue
    counts = []
    for qid in CLEAN[split]["queries"]:
        counts.append(len(CLEAN[split]["queries"][qid]["relevant"]))
    arr = np.array(counts)
    pool_size = len(CLEAN[split]["cands"])
    print("  %s: mean %.2f | median %.0f | max %d | share with >7: %.1f%% | share with 1: %.1f%% | pool %d (%.3f%% relevant per query)" % (
        split, arr.mean(), np.median(arr), arr.max(), 100.0 * (arr > 7).mean(), 100.0 * (arr == 1).mean(),
        pool_size, 100.0 * arr.mean() / max(pool_size, 1)))
    tp_max = 0
    for c in counts:
        tp_max += min(c, TOP_K_EVAL)
    nq = len(counts)
    p = tp_max / (TOP_K_EVAL * nq)
    r = tp_max / arr.sum()
    f1 = 2 * p * r / (p + r)
    print("     oracle Micro-F1@%d = %.4f (P %.4f, R %.4f)" % (TOP_K_EVAL, f1, p, r))

# candidate popularity in train: how many queries share the same gold candidate
if "train" in CLEAN:
    pop = Counter()
    for qid in CLEAN["train"]["queries"]:
        for r in CLEAN["train"]["queries"][qid]["relevant"]:
            pop[r] += 1
    pop_values = list(pop.values())
    print("\ntrain gold candidates: %d distinct of pool %d (%.1f%% of the pool is ever relevant)" % (
        len(pop), len(CLEAN["train"]["cands"]), 100.0 * len(pop) / len(CLEAN["train"]["cands"])))
    print("  times a gold candidate is shared across queries: mean %.2f | max %d | share used by >1 query: %.1f%%" % (
        np.mean(pop_values), np.max(pop_values), 100.0 * np.mean(np.array(pop_values) > 1)))

    # twin check: gold candidates that have an identical-text twin in the pool
    h_to_ids = defaultdict(list)
    for cid in CLEAN["train"]["cands"]:
        h_to_ids[text_hash(CLEAN["train"]["cands"][cid]["sentences"])].append(cid)
    twin_gold = 0
    twin_not_gold = 0
    for qid in CLEAN["train"]["queries"]:
        rel = set(CLEAN["train"]["queries"][qid]["relevant"])
        for r in rel:
            h = text_hash(CLEAN["train"]["cands"][r]["sentences"])
            for t in h_to_ids[h]:
                if t == r:
                    continue
                if t in rel:
                    twin_gold += 1
                else:
                    twin_not_gold += 1
    print("  gold candidates' identical-text twins: also gold %d | NOT gold %d" % (twin_gold, twin_not_gold))

    # does gold ever equal the query itself (same id or same text)?
    self_gold = 0
    same_text_gold = 0
    for qid in CLEAN["train"]["queries"]:
        rel = CLEAN["train"]["queries"][qid]["relevant"]
        qh = text_hash(CLEAN["train"]["queries"][qid]["sentences"])
        for r in rel:
            if r == qid:
                self_gold += 1
            elif text_hash(CLEAN["train"]["cands"][r]["sentences"]) == qh:
                same_text_gold += 1
    print("  gold equal to the query (same id): %d | same text, other id: %d" % (self_gold, same_text_gold))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
if "train" in CLEAN:
    counts = []
    for qid in CLEAN["train"]["queries"]:
        counts.append(len(CLEAN["train"]["queries"][qid]["relevant"]))
    axes[0].hist(counts, bins=range(1, max(counts) + 2))
    axes[0].set_title("relevant candidates per train query")
    axes[1].hist(np.log10(np.array(STATS["train"]["q_words"]) + 1), bins=40, alpha=0.6, label="query")
    axes[1].hist(np.log10(np.array(STATS["train"]["c_words"]) + 1), bins=40, alpha=0.6, label="candidate")
    axes[1].set_title("log10 words (train)")
    axes[1].legend()
    axes[2].hist(pop_values, bins=range(1, max(pop_values) + 2))
    axes[2].set_title("queries sharing a gold candidate")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "basic_stats.png"), dpi=110)
plt.show()

## 9. Lexical baselines (BM25) + candidate recall curve
BM25 is fitted on each split's own pool (no labels, no outside data). It gives: a baseline Micro-F1@7, the recall@k ceiling for any reranker, and the hard-negative pool for training.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp


def doc_text(sentences):
    return "\n".join(sentences)


def build_bm25(pool_ids, pool_texts):
    vec = CountVectorizer(token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", lowercase=True,
                          stop_words="english", min_df=2, max_df=0.7, dtype=np.float32)
    X = vec.fit_transform(pool_texts).tocsr()
    n_docs = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float64)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
    avgdl = max(dl.mean(), 1.0)
    row_of_nz = np.repeat(np.arange(n_docs), np.diff(X.indptr))
    tf = X.data.astype(np.float64)
    denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * dl[row_of_nz] / avgdl)
    weights = idf[X.indices] * tf * (BM25_K1 + 1.0) / denom
    W = sp.csr_matrix((weights.astype(np.float32), X.indices.copy(), X.indptr.copy()), shape=X.shape)
    return {"vec": vec, "WT": W.T.tocsr(), "ids": list(pool_ids), "id_to_pos": {c: i for i, c in enumerate(pool_ids)}}


def bm25_rank(index, query_ids, query_texts, mask_hashes=None, query_hashes=None, top_n=100):
    # returns dict qid -> (list of cand ids, list of scores), self / identical-text docs masked out
    ids = index["ids"]
    n_pool = len(ids)
    top_n = min(top_n, n_pool)
    out = {}
    for start in range(0, len(query_ids), BM25_QUERY_BATCH):
        batch_ids = query_ids[start:start + BM25_QUERY_BATCH]
        batch_texts = query_texts[start:start + BM25_QUERY_BATCH]
        Q = index["vec"].transform(batch_texts).tocsr()
        Q.data = np.log1p(Q.data)
        scores = (Q @ index["WT"]).toarray()
        for i, qid in enumerate(batch_ids):
            row = scores[i]
            if qid in index["id_to_pos"]:
                row[index["id_to_pos"][qid]] = -1e9
            if mask_hashes is not None:
                same = mask_hashes.get(query_hashes[qid], [])
                for cid in same:
                    row[index["id_to_pos"][cid]] = -1e9
            part = np.argpartition(-row, top_n - 1)[:top_n]
            part = part[np.argsort(-row[part])]
            out[qid] = ([ids[j] for j in part], [float(row[j]) for j in part])
    return out


def micro_prf(pred, gold, k=7):
    tp = 0
    n_gold = 0
    n_pred = 0
    for qid in gold:
        ranked = pred.get(qid, [])[:k]
        g = set(gold[qid])
        n_gold += len(g)
        n_pred += k
        for c in ranked:
            if c in g:
                tp += 1
    p = tp / max(n_pred, 1)
    r = tp / max(n_gold, 1)
    f1 = 0.0
    if p + r > 0:
        f1 = 2 * p * r / (p + r)
    return p, r, f1


def recall_at(pred, gold, k):
    tp = 0
    n_gold = 0
    for qid in gold:
        g = set(gold[qid])
        n_gold += len(g)
        for c in pred.get(qid, [])[:k]:
            if c in g:
                tp += 1
    return tp / max(n_gold, 1)


BM25_RANK = {}
BM25_INDEX = {}
for split in PRESENT_SPLITS:
    t0 = time.time()
    pool_ids = list(CLEAN[split]["cands"].keys())
    pool_texts = []
    h_to_ids = defaultdict(list)
    for cid in pool_ids:
        pool_texts.append(doc_text(CLEAN[split]["cands"][cid]["sentences"]))
        h_to_ids[text_hash(CLEAN[split]["cands"][cid]["sentences"])].append(cid)
    index = build_bm25(pool_ids, pool_texts)
    BM25_INDEX[split] = index

    q_ids = list(CLEAN[split]["queries"].keys())
    q_texts = []
    q_hashes = {}
    for qid in q_ids:
        q_texts.append(doc_text(CLEAN[split]["queries"][qid]["sentences"]))
        q_hashes[qid] = text_hash(CLEAN[split]["queries"][qid]["sentences"])
    BM25_RANK[split] = bm25_rank(index, q_ids, q_texts, mask_hashes=h_to_ids, query_hashes=q_hashes, top_n=BM25_TOP_N)
    print("%s: BM25 ranked %d queries over a pool of %d in %.0fs (vocab %d)" % (
        split, len(q_ids), len(pool_ids), time.time() - t0, len(index["vec"].vocabulary_)))

    if split != "test":
        gold = {}
        pred = {}
        for qid in q_ids:
            gold[qid] = CLEAN[split]["queries"][qid]["relevant"]
            pred[qid] = BM25_RANK[split][qid][0]
        p, r, f1 = micro_prf(pred, gold, TOP_K_EVAL)
        print("   BM25 Micro-F1@%d = %.4f (P %.4f, R %.4f)" % (TOP_K_EVAL, f1, p, r))
        for k in [7, 20, 50, 100]:
            if k <= BM25_TOP_N:
                print("   recall@%-3d = %.3f" % (k, recall_at(pred, gold, k)))

In [ ]:
# Which part of a long case carries the matching signal? (head / tail / head+tail / full)
def make_view(sentences, mode, n_words):
    words = " ".join(sentences).split()
    if mode == "full":
        return " ".join(words)
    if mode == "head":
        return " ".join(words[:n_words])
    if mode == "tail":
        return " ".join(words[-n_words:])
    half = n_words // 2
    return " ".join(words[:half] + words[-half:])


VIEW_RESULTS = {}
if "dev" in CLEAN:
    dev_q_ids = list(CLEAN["dev"]["queries"].keys())
    rng = np.random.RandomState(SEED)
    if len(dev_q_ids) > ANALYSIS_QUERIES:
        dev_q_ids = list(rng.choice(dev_q_ids, size=ANALYSIS_QUERIES, replace=False))
    pool_ids = list(CLEAN["dev"]["cands"].keys())
    gold = {}
    for qid in dev_q_ids:
        gold[qid] = CLEAN["dev"]["queries"][qid]["relevant"]

    for mode in ["full", "head", "tail", "headtail"]:
        t0 = time.time()
        pool_texts = []
        for cid in pool_ids:
            pool_texts.append(make_view(CLEAN["dev"]["cands"][cid]["sentences"], mode, VIEW_WORDS))
        q_texts = []
        for qid in dev_q_ids:
            q_texts.append(make_view(CLEAN["dev"]["queries"][qid]["sentences"], mode, VIEW_WORDS))
        index = build_bm25(pool_ids, pool_texts)
        ranked = bm25_rank(index, dev_q_ids, q_texts, top_n=TOP_K_EVAL)
        pred = {}
        for qid in dev_q_ids:
            pred[qid] = ranked[qid][0]
        p, r, f1 = micro_prf(pred, gold, TOP_K_EVAL)
        VIEW_RESULTS[mode] = f1
        print("view %-9s (%s words) BM25 Micro-F1@%d = %.4f  (%.0fs)" % (
            mode, "all" if mode == "full" else str(VIEW_WORDS), TOP_K_EVAL, f1, time.time() - t0))
    print("\nIf head / tail / head+tail are close to full, a single 1024-token window may be enough;")
    print("if full is far ahead, chunked / hierarchical encoding is worth the cost.")
else:
    print("No dev split, skipping the view analysis.")

if "dev" in BM25_RANK:
    gold = {}
    pred = {}
    for qid in CLEAN["dev"]["queries"]:
        gold[qid] = CLEAN["dev"]["queries"][qid]["relevant"]
        pred[qid] = BM25_RANK["dev"][qid][0]
    ks = [1, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100]
    rec = []
    for k in ks:
        if k <= BM25_TOP_N:
            rec.append(recall_at(pred, gold, k))
    plt.figure(figsize=(5, 3.2))
    plt.plot(ks[:len(rec)], rec, marker="o")
    plt.xlabel("k")
    plt.ylabel("recall@k")
    plt.title("BM25 candidate recall (dev)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "bm25_recall_curve.png"), dpi=110)
    plt.show()

## 10. Load the Phase-1 tokenizer

In [ ]:
from transformers import AutoTokenizer
os.environ["TOKENIZERS_PARALLELISM"] = "true"


def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    paths = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    folders = []
    for p in paths:
        folder = os.path.dirname(p)
        if "checkpoint-" not in folder:
            folders.append(folder)
    folders = sorted(set(folders))
    if len(folders) == 0:
        raise FileNotFoundError("No tokenizer.json under /kaggle/input. Attach your tokenizer dataset (Add Input).")
    for folder in folders:
        if os.path.exists(os.path.join(folder, "trained_on.json")):
            return folder
    return folders[0]


tok_dir = find_tokenizer_dir()
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
tokenizer.model_max_length = 10 ** 9
VOCAB_SIZE = len(tokenizer)
print("Tokenizer:", tok_dir, "| vocab size:", VOCAB_SIZE)


def find_sep_id(tok):
    if tok.eos_token_id is not None:
        return int(tok.eos_token_id)
    for name in ["<eos>", "</s>", "<|endoftext|>", "<EOS>"]:
        tid = tok.convert_tokens_to_ids(name)
        if tid is not None and tid != tok.unk_token_id:
            return int(tid)
    raise ValueError("Could not find an EOS-like token to use as separator.")


SEP_ID = find_sep_id(tokenizer)
if tokenizer.pad_token_id is not None:
    PAD_ID = int(tokenizer.pad_token_id)
else:
    PAD_ID = SEP_ID
print("SEP (EOS) id:", SEP_ID, "| PAD id:", PAD_ID)
if PAD_ID == SEP_ID:
    print("PAD and SEP share an id. In training mask by length, never by token id.")

if VOCAB_SIZE < 65536:
    TOKEN_DTYPE = np.uint16
else:
    TOKEN_DTYPE = np.int32
print("Token arrays are stored as", np.dtype(TOKEN_DTYPE).name)

## 11. Tokenise every query and candidate
Full documents are stored once as compact arrays. Chunk / head+tail views are cut from them in `pcr_utils.py`, so no information is thrown away here.

In [ ]:
def tokenize_texts(texts, batch_size=32):
    arrays = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, add_special_tokens=False, return_attention_mask=False, truncation=False)
        for ids in enc["input_ids"]:
            arrays.append(np.array(ids, dtype=TOKEN_DTYPE))
    return arrays


TOK = {}
for split in PRESENT_SPLITS:
    t0 = time.time()
    TOK[split] = {"queries": {}, "cands": {}}
    for name in ["queries", "cands"]:
        ids_list = list(CLEAN[split][name].keys())
        texts = []
        for i in ids_list:
            texts.append(doc_text(CLEAN[split][name][i]["sentences"]))
        arrays = tokenize_texts(texts)
        for i, arr in zip(ids_list, arrays):
            TOK[split][name][i] = arr
    print("%s tokenised in %.0fs" % (split, time.time() - t0))


def n_chunks(length):
    if length <= CHUNK_SIZE:
        return 1
    step = CHUNK_SIZE - CHUNK_OVERLAP
    return 1 + int(np.ceil((length - CHUNK_SIZE) / step))


TOKEN_STATS = {}
for split in PRESENT_SPLITS:
    for name in ["queries", "cands"]:
        lengths = []
        words = []
        for i in TOK[split][name]:
            lengths.append(len(TOK[split][name][i]))
            words.append(words_of_doc(CLEAN[split][name][i]["sentences"]))
        arr = np.array(lengths)
        chunks = np.array([n_chunks(L) for L in lengths])
        print("%s %-8s tokens: mean %.0f | median %.0f | p90 %.0f | p99 %.0f | max %d | tokens/word %.2f" % (
            split, name, arr.mean(), np.median(arr), np.percentile(arr, 90), np.percentile(arr, 99), arr.max(),
            arr.sum() / max(sum(words), 1)))
        print("    fits 1024: %.1f%% | fits 2048: %.1f%% | fits 4096: %.1f%% | chunks(1024/128) mean %.1f | >%d chunks: %.1f%%" % (
            100.0 * (arr <= 1024).mean(), 100.0 * (arr <= 2048).mean(), 100.0 * (arr <= 4096).mean(),
            chunks.mean(), MAX_CHUNKS, 100.0 * (chunks > MAX_CHUNKS).mean()))
        TOKEN_STATS[(split, name)] = {"mean": float(arr.mean()), "median": float(np.median(arr)), "p99": float(np.percentile(arr, 99))}

# unknown-token / round trip sanity on a few docs
check_split = PRESENT_SPLITS[0]
check_ids = list(TOK[check_split]["queries"].keys())[:3]
for i in check_ids:
    ids = TOK[check_split]["queries"][i][:60].astype(np.int64).tolist()
    print("sample", i, "->", repr(tokenizer.decode(ids)[:120]))

## 12. Hard negatives
Top BM25 candidates that are **not** gold. Note: in legal PCR some of these are unlabeled-but-relevant cases, so mixing in easier negatives is safer than using only the very top.

In [ ]:
HARD_NEG = {}
for split in ["train", "dev"]:
    if split not in CLEAN:
        continue
    HARD_NEG[split] = {}
    n_avail = []
    gold_ranks = []
    for qid in CLEAN[split]["queries"]:
        gold = set(CLEAN[split]["queries"][qid]["relevant"])
        ranked = BM25_RANK[split][qid][0]
        negs = []
        for rank, c in enumerate(ranked):
            if c in gold:
                gold_ranks.append(rank + 1)
            else:
                negs.append(c)
        HARD_NEG[split][qid] = negs
        n_avail.append(len(negs))
    print("%s: hard negatives per query (from top-%d): mean %.1f | min %d" % (split, BM25_TOP_N, np.mean(n_avail), np.min(n_avail)))
    print("    median BM25 rank of a gold candidate (when found in top-%d): %.0f" % (BM25_TOP_N, np.median(gold_ranks)))

## 13. Folds (train queries)
Queries that share a gold candidate (or have identical text) are grouped, so a fold never leaks a gold candidate across train/val. Dev stays a separate, untouched validation set.

In [ ]:
FOLD_OF_QUERY = {}
if "train" in CLEAN:
    q_ids = list(CLEAN["train"]["queries"].keys())
    parent = {}
    for q in q_ids:
        parent[q] = q

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra = find(a)
        rb = find(b)
        if ra != rb:
            parent[ra] = rb

    owner_of_gold = {}
    owner_of_text = {}
    for q in q_ids:
        for r in CLEAN["train"]["queries"][q]["relevant"]:
            if r in owner_of_gold:
                union(q, owner_of_gold[r])
            else:
                owner_of_gold[r] = q
        h = text_hash(CLEAN["train"]["queries"][q]["sentences"])
        if h in owner_of_text:
            union(q, owner_of_text[h])
        else:
            owner_of_text[h] = q

    groups = defaultdict(list)
    for q in q_ids:
        groups[find(q)].append(q)
    group_list = sorted(groups.values(), key=len, reverse=True)
    print("train queries:", len(q_ids), "| groups:", len(group_list), "| largest group:", len(group_list[0]))

    fold_sizes = [0] * N_FOLDS
    rng = np.random.RandomState(SEED)
    # shuffle groups of equal size so folds are not ordered by id
    order = list(range(len(group_list)))
    rng.shuffle(order)
    order.sort(key=lambda g: -len(group_list[g]))
    for g in order:
        fold = int(np.argmin(fold_sizes))
        for q in group_list[g]:
            FOLD_OF_QUERY[q] = fold
        fold_sizes[fold] += len(group_list[g])
    print("fold sizes:", fold_sizes)
    for f in range(N_FOLDS):
        counts = []
        for q in q_ids:
            if FOLD_OF_QUERY[q] == f:
                counts.append(len(CLEAN["train"]["queries"][q]["relevant"]))
        print("  fold %d: mean relevant per query %.2f" % (f, np.mean(counts)))

## 14. Save: cleaned JSONs, tokenised bundle, helper file, BM25 baseline prediction

In [ ]:
CLEAN_DIR = os.path.join(OUT_DIR, "clean_json")
os.makedirs(CLEAN_DIR, exist_ok=True)
for split in PRESENT_SPLITS:
    q_out = {}
    for qid in CLEAN[split]["queries"]:
        entry = {"text": CLEAN[split]["queries"][qid]["sentences"]}
        if CLEAN[split]["queries"][qid]["relevant"] is not None:
            entry["relevant_candidates"] = CLEAN[split]["queries"][qid]["relevant"]
        q_out[qid] = entry
    c_out = {}
    for cid in CLEAN[split]["cands"]:
        c_out[cid] = {"text": CLEAN[split]["cands"][cid]["sentences"]}
    with open(os.path.join(CLEAN_DIR, split + ".json"), "w") as f:
        json.dump(q_out, f)
    with open(os.path.join(CLEAN_DIR, split + "_candidates.json"), "w") as f:
        json.dump(c_out, f)

BUNDLE = {
    "splits": PRESENT_SPLITS,
    "tokens": TOK,
    "gold": {},
    "pool_ids": {},
    "query_ids": {},
    "bm25_rank": BM25_RANK,
    "hard_neg": HARD_NEG,
    "fold_of_query": FOLD_OF_QUERY,
    "sep_id": SEP_ID,
    "pad_id": PAD_ID,
    "vocab_size": VOCAB_SIZE,
    "token_dtype": np.dtype(TOKEN_DTYPE).name,
    "tokenizer_dir": tok_dir,
    "config": {
        "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP, "max_chunks": MAX_CHUNKS,
        "top_k_eval": TOP_K_EVAL, "seed": SEED, "n_folds": N_FOLDS,
    },
    "view_results": VIEW_RESULTS,
}
for split in PRESENT_SPLITS:
    BUNDLE["pool_ids"][split] = list(CLEAN[split]["cands"].keys())
    BUNDLE["query_ids"][split] = list(CLEAN[split]["queries"].keys())
    BUNDLE["gold"][split] = {}
    if split != "test":
        for qid in CLEAN[split]["queries"]:
            BUNDLE["gold"][split][qid] = CLEAN[split]["queries"][qid]["relevant"]

bundle_path = os.path.join(OUT_DIR, "tokenized_pcr.pkl")
with open(bundle_path, "wb") as f:
    pickle.dump(BUNDLE, f, protocol=4)
print("saved", bundle_path, "| %.1f MB" % (os.path.getsize(bundle_path) / 1e6))

In [ ]:
HELPER_SRC = r"""
import json
import pickle
import numpy as np


def load_bundle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def chunk_ids(ids, size=1024, overlap=128, max_chunks=16):
    # overlapping chunks over the whole document; if too long, keeps evenly spaced chunks
    ids = np.asarray(ids)
    if len(ids) <= size:
        return [ids]
    step = size - overlap
    starts = list(range(0, len(ids) - overlap, step))
    chunks = []
    for s in starts:
        chunk = ids[s:s + size]
        if len(chunk) > overlap or len(chunks) == 0:
            chunks.append(chunk)
    if len(chunks) > max_chunks:
        picks = np.linspace(0, len(chunks) - 1, max_chunks).round().astype(int)
        chosen = []
        for p in picks:
            chosen.append(chunks[p])
        chunks = chosen
    return chunks


def head_tail_ids(ids, max_len=1024, head_frac=0.5):
    ids = np.asarray(ids)
    if len(ids) <= max_len:
        return ids
    n_head = int(max_len * head_frac)
    n_tail = max_len - n_head
    return np.concatenate([ids[:n_head], ids[-n_tail:]])


def pad_batch(list_of_arrays, pad_id, max_len=None):
    # returns (ids [B, L] int64, lengths [B]); mask by length, never by token id
    lengths = []
    for a in list_of_arrays:
        lengths.append(len(a))
    L = max(lengths)
    if max_len is not None:
        L = min(L, max_len)
    out = np.full((len(list_of_arrays), L), pad_id, dtype=np.int64)
    out_len = np.zeros(len(list_of_arrays), dtype=np.int64)
    for i, a in enumerate(list_of_arrays):
        a = a[:L]
        out[i, :len(a)] = a.astype(np.int64)
        out_len[i] = len(a)
    return out, out_len


def micro_prf_at_k(pred, gold, k=7):
    tp = 0
    n_gold = 0
    n_pred = 0
    for qid in gold:
        ranked = pred.get(qid, [])[:k]
        g = set(gold[qid])
        n_gold += len(g)
        n_pred += k
        for c in ranked:
            if c in g:
                tp += 1
    p = tp / max(n_pred, 1)
    r = tp / max(n_gold, 1)
    f1 = 0.0
    if p + r > 0:
        f1 = 2 * p * r / (p + r)
    return {"precision": p, "recall": r, "f1": f1}


def rank_from_scores(scores, cand_ids, query_id=None, k=7, ban_ids=None):
    # scores: 1-D array aligned with cand_ids; query itself (and any ban_ids) is excluded
    scores = np.array(scores, dtype=np.float64)
    ban = set(ban_ids or [])
    if query_id is not None:
        ban.add(query_id)
    for i, c in enumerate(cand_ids):
        if c in ban:
            scores[i] = -1e18
    order = np.argsort(-scores)[:k]
    result = []
    for j in order:
        result.append(cand_ids[j])
    return result


def validate_prediction(pred, bundle, split="test", k=7):
    problems = []
    pool = set(bundle["pool_ids"][split])
    for qid in bundle["query_ids"][split]:
        if qid not in pred:
            problems.append("missing query " + qid)
            continue
        ranked = pred[qid]
        if len(ranked) != k:
            problems.append("query %s has %d ids (need %d)" % (qid, len(ranked), k))
        if len(set(ranked)) != len(ranked):
            problems.append("query %s has duplicate ids" % qid)
        for c in ranked:
            if c not in pool:
                problems.append("query %s: %s not in %s pool" % (qid, c, split))
                break
    for qid in pred:
        if qid not in set(bundle["query_ids"][split]):
            problems.append("unknown query " + qid)
    return problems


def write_prediction(pred, bundle, path, split="test", k=7):
    problems = validate_prediction(pred, bundle, split, k)
    if len(problems) > 0:
        raise ValueError("Invalid prediction file: " + "; ".join(problems[:5]))
    with open(path, "w") as f:
        json.dump(pred, f)
    print("wrote", path, "for", len(pred), "queries")


def bm25_prediction(bundle, split="test", k=7):
    pred = {}
    for qid in bundle["query_ids"][split]:
        pred[qid] = bundle["bm25_rank"][split][qid][0][:k]
    return pred
"""

helper_path = os.path.join(OUT_DIR, "pcr_utils.py")
with open(helper_path, "w") as f:
    f.write(HELPER_SRC)
print("saved", helper_path)

# round-trip test of the helper file + baseline submission
sys.path.insert(0, OUT_DIR)
import importlib
import pcr_utils
importlib.reload(pcr_utils)

b = pcr_utils.load_bundle(bundle_path)
if "dev" in b["splits"]:
    pred = pcr_utils.bm25_prediction(b, "dev")
    print("helper check, dev BM25:", pcr_utils.micro_prf_at_k(pred, b["gold"]["dev"], 7))
if "test" in b["splits"]:
    pred = pcr_utils.bm25_prediction(b, "test")
    pcr_utils.write_prediction(pred, b, os.path.join(OUT_DIR, "baseline_bm25_prediction.json"), "test", 7)
else:
    print("No test split yet: re-run this notebook when the test files are released.")

example_id = b["query_ids"]["train"][0]
example_chunks = pcr_utils.chunk_ids(b["tokens"]["train"]["queries"][example_id], CHUNK_SIZE, CHUNK_OVERLAP, MAX_CHUNKS)
print("chunk helper: query", example_id, "->", len(example_chunks), "chunks; first lengths", [len(c) for c in example_chunks[:4]])

## 15. Summary

In [ ]:
print("Outputs in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  %-34s %.1f MB" % (name, os.path.getsize(full) / 1e6))
    else:
        print("  %-34s (folder)" % (name + "/"))
print("\nSplits:", PRESENT_SPLITS)
for split in PRESENT_SPLITS:
    print("  %-5s queries %6d | pool %6d" % (split, len(CLEAN[split]["queries"]), len(CLEAN[split]["cands"])))
print("\nView analysis (BM25 Micro-F1@7 on dev sample):", {k: round(v, 4) for k, v in VIEW_RESULTS.items()})